# P3 — ML attempt: can a model beat `funding_momentum`?

**Slug**: `crypto_intraday/30_ml`

**Status**: in_progress

**Research question**: Train logistic regression, ridge regression, and LightGBM on the leak-safe feature catalog (intraday + funding + cross-asset) with walk-forward CV. Convert OOS predictions to weights, backtest with stress costs. **Promote ML only if it beats `funding_momentum` OOS net Sharpe (1.56 at perp_stress on Q1+Q2 2024).** Otherwise reject — the spec is explicit on this.

**Setup**:
- 1h sampling, Q1+Q2 2024, perp (same as P2).
- Walk-forward expanding splits: train_size=60D, val_size=30D, step=30D — 4 folds.
- `Standardizer` fit on TRAIN ONLY per fold; applied to val.
- Per-symbol models (BTC, ETH).
- Target: binary direction over next 1h (logistic / LightGBM) or continuous return (ridge).
- Evaluation: concatenate OOS predictions, form weight series, backtest with `perp_stress` costs.

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression, Ridge
import lightgbm as lgb

from alpha_lab.backtest.metrics import summary
from alpha_lab.backtest.vector import run_backtest
from alpha_lab.data.holdout import PMHoldout, access_summary_for_report, safe_forward_returns
from alpha_lab.data.loaders import binance_vision as bv
from alpha_lab.features import intraday as f
from alpha_lab.features.transforms import Standardizer
from alpha_lab.ml.cv import WalkForwardSplit
from alpha_lab.utils.config import load_config

pd.set_option('display.float_format', '{:,.4f}'.format)
pd.set_option('display.width', 200)

In [ ]:
cfg = load_config('crypto_intraday')
holdout = PMHoldout.from_config('crypto_intraday')
SYMBOLS = ['BTCUSDT', 'ETHUSDT']
BARS_PER_YEAR_1H = 365 * 24
RES_START, RES_END = '2024-01-01', '2024-07-01'

panel_1m = bv.load_klines(SYMBOLS, '1m', start=RES_START, end=RES_END, market='perp', holdout=holdout)
def _agg(df, off):
    return df.resample(off, label='right', closed='right').agg({
        'open':'first','high':'max','low':'min','close':'last',
        'volume':'sum','quote_volume':'sum','n_trades':'sum','taker_buy_base':'sum',
    }).dropna(how='all')
ohlcv_1h = {s: _agg(df, '1h') for s, df in panel_1m.frames.items()}
prices_1h = pd.DataFrame({s: o['close'] for s, o in ohlcv_1h.items()}).dropna()
common_idx = prices_1h.index
ohlcv_1h = {s: o.loc[common_idx] for s, o in ohlcv_1h.items()}
funding = bv.load_funding(SYMBOLS, start=RES_START, end=RES_END, holdout=holdout)
print(f'1h bars: {len(common_idx)}')

## Build per-symbol feature matrix

All features lagged / rolling (leak-safe). Funding features forward-filled onto 1h grid.

In [ ]:
funding_z_grid = f.funding_zscore(funding, window=30).reindex(common_idx, method='ffill')
funding_cum_grid = f.funding_cumulative(funding, window=10).reindex(common_idx, method='ffill')
funding_level_grid = funding.reindex(common_idx, method='ffill')

def build_feature_panel(sym: str) -> pd.DataFrame:
    o = ohlcv_1h[sym]
    c = o['close']
    feats = {
        'ret_1h':       f.log_return(c, 1),
        'ret_4h':       f.log_return(c, 4),
        'ret_24h':      f.log_return(c, 24),
        'rvol_24h':     f.realized_vol_close(c, 24),
        'rvol_park_24h': f.realized_vol_parkinson(o['high'], o['low'], 24),
        'vol_z_24h':    f.volume_zscore(o['volume'], 24),
        'taker_imb_24h': f.rolling_taker_imbalance(o['taker_buy_base'], o['volume'], 24),
        'distma_48h':   f.distance_from_ma(c, 48),
        'breakout_48h': f.breakout_distance(c, 48),
        'atr_14h':      f.atr(o['high'], o['low'], c, 14),
        'rsi_14h':      f.rsi(c, 14),
        'bollb_20h':    f.bollinger_pct_b(c, 20),
        'donch_20h':    f.donchian_position(o['high'], o['low'], c, 20),
        'macd_hist':    f.macd(c)['hist'],
        'hour_of_day':  f.time_of_day_hours(c.index),
        'day_of_week':  f.day_of_week(c.index).astype(float),
        'funding_z':    funding_z_grid[sym],
        'funding_cum':  funding_cum_grid[sym],
        'funding_lvl':  funding_level_grid[sym],
    }
    return pd.DataFrame(feats)

features_per_sym = {sym: build_feature_panel(sym) for sym in SYMBOLS}
for sym, F in features_per_sym.items():
    print(f'{sym}: shape={F.shape}, cols={len(F.columns)}')

## Labels: forward 1h direction (for logistic / lgb), forward 1h return (for ridge)

`safe_forward_returns` masks rows whose forward window crosses the holdout boundary (no-op here since we're in 2024).

In [ ]:
fwd_returns = {sym: safe_forward_returns(prices_1h[sym].pct_change().fillna(0), 1, holdout=holdout) for sym in SYMBOLS}
fwd_direction = {sym: (fwd_returns[sym] > 0).astype(int) for sym in SYMBOLS}
print('Base rate (P(up)):')
for sym in SYMBOLS:
    print(f'  {sym}: {fwd_direction[sym].mean():.4f}')

## Walk-forward train + predict

Per-symbol models, OOS predictions concatenated across folds, then combined into a 2-column weight matrix and backtested.

In [ ]:
wf = WalkForwardSplit(
    train_size='60D', val_size='30D', step='30D', mode='expanding',
)

def walk_forward_predict(model_factory, X, y, *, is_regression=False) -> pd.Series:
    """Run walk-forward, return OOS predictions aligned to X.index. Train rows with
    any NaN are dropped (per-fold)."""
    preds = pd.Series(np.nan, index=X.index)
    fold_count = 0
    for split in wf.split(X.index):
        X_tr = X.loc[split.train].dropna()
        y_tr = y.loc[X_tr.index]
        X_va = X.loc[split.val].dropna()
        if len(X_tr) < 200 or len(X_va) < 50:
            continue
        # Drop val rows whose label is NaN (e.g. last-bar fwd-return is NaN)
        y_va_idx = y.loc[X_va.index].dropna().index
        X_va = X_va.loc[y_va_idx]
        if len(X_va) == 0:
            continue
        sc = Standardizer(mode='per_column').fit(X_tr)
        X_tr_s = sc.transform(X_tr).values
        X_va_s = sc.transform(X_va).values
        # Clip residual NaN/inf (rare)
        X_tr_s = np.nan_to_num(X_tr_s)
        X_va_s = np.nan_to_num(X_va_s)
        model = model_factory()
        model.fit(X_tr_s, y_tr.values)
        if is_regression:
            preds.loc[X_va.index] = model.predict(X_va_s)
        else:
            # probability of class 1 (up)
            preds.loc[X_va.index] = model.predict_proba(X_va_s)[:, 1]
        fold_count += 1
    return preds, fold_count

In [ ]:
models = {
    'logistic': (lambda: LogisticRegression(C=1.0, max_iter=500), False),
    'ridge':    (lambda: Ridge(alpha=1.0), True),
    'lightgbm': (lambda: lgb.LGBMClassifier(n_estimators=200, max_depth=4, learning_rate=0.05, verbose=-1), False),
}

all_preds: dict[str, dict[str, pd.Series]] = {m: {} for m in models}
fold_counts: dict[str, int] = {m: 0 for m in models}
for sym in SYMBOLS:
    X_full = features_per_sym[sym]
    y_dir = fwd_direction[sym]
    y_ret = fwd_returns[sym]
    for name, (factory, is_reg) in models.items():
        y = y_ret if is_reg else y_dir
        # Align X and y
        common = X_full.dropna().index.intersection(y.dropna().index)
        X = X_full.loc[common]
        y_al = y.loc[common]
        preds, fc = walk_forward_predict(factory, X, y_al, is_regression=is_reg)
        all_preds[name][sym] = preds
        fold_counts[name] = fc

for name in models:
    print(f'{name}: {fold_counts[name]} folds completed')

## Convert predictions to weights and backtest

In [ ]:
from dataclasses import dataclass
@dataclass(frozen=True)
class CostScenario:
    name: str
    fee_bps: float
    slip_bps: dict[str, float]
    use_funding: bool

scenarios = []
for key in ['zero', 'perp_base', 'perp_stress']:
    block = cfg['costs'][key]
    scenarios.append(CostScenario(
        name=key,
        fee_bps=float(block['fee_bps_per_side']),
        slip_bps={s: float(block['slippage_bps_per_side'][s]) for s in SYMBOLS},
        use_funding=bool(block.get('include_funding', False)),
    ))

def preds_to_weights(preds: dict[str, pd.Series], is_classifier: bool, threshold: float = 0.5) -> pd.DataFrame:
    cols = {}
    for sym in SYMBOLS:
        p = preds[sym]
        if is_classifier:
            w = pd.Series(0.0, index=p.index)
            w[p > threshold] = 0.5
            w[p < 1 - threshold] = -0.5
        else:
            # Ridge: predict continuous return. Sign-only weight.
            w = np.sign(p).fillna(0.0) * 0.5
        cols[sym] = w
    return pd.DataFrame(cols).reindex(common_idx).fillna(0.0)

In [ ]:
rows = []
for name, (_, is_reg) in models.items():
    weights = preds_to_weights(all_preds[name], is_classifier=(not is_reg), threshold=0.55)
    for sc in scenarios:
        res = run_backtest(
            weights, prices_1h,
            costs_bps=sc.fee_bps, slippage_bps=sc.slip_bps,
            funding=funding if sc.use_funding else None,
            bars_per_year=BARS_PER_YEAR_1H,
        )
        stats = summary(res.returns, periods=BARS_PER_YEAR_1H)
        rows.append({
            'model': name,
            'scenario': sc.name,
            'gross_total': float((1 + res.gross_returns).prod() - 1),
            'net_total':   float((1 + res.returns).prod() - 1),
            'Sharpe':      stats.get('Sharpe', float('nan')),
            'turnover_sum': float(res.turnover.sum()),
        })
ml_results = pd.DataFrame(rows)
print(ml_results.pivot(index='model', columns='scenario', values='net_total')[['zero','perp_base','perp_stress']])

In [ ]:
print('Annualized Sharpe:')
print(ml_results.pivot(index='model', columns='scenario', values='Sharpe')[['zero','perp_base','perp_stress']])
print()
print('Turnover:')
print(ml_results.pivot(index='model', columns='scenario', values='turnover_sum')[['perp_base']])

## Comparison vs `funding_momentum` baseline

From P2-strategies decision: `funding_momentum` net total at perp_stress = +0.4082, Sharpe = 1.5564 over the same Q1+Q2 2024 window. **ML promotion requires beating this.**

In [ ]:
FUND_MOM_STRESS_NET = 0.4082
FUND_MOM_STRESS_SHARPE = 1.5564

stress = ml_results[ml_results.scenario == 'perp_stress'].set_index('model')
print('ML net & Sharpe at perp_stress:')
for name in models:
    n = stress.loc[name, 'net_total']
    s = stress.loc[name, 'Sharpe']
    beats_net = n > FUND_MOM_STRESS_NET
    beats_shp = (not pd.isna(s)) and s > FUND_MOM_STRESS_SHARPE
    print(f'  {name:10s}: net={n:+.4f} (vs {FUND_MOM_STRESS_NET:+.4f}, beats={beats_net}), '
          f'Sharpe={s:+.4f} (vs {FUND_MOM_STRESS_SHARPE:+.4f}, beats={beats_shp})')

## PM holdout audit

In [ ]:
audit = access_summary_for_report()
for k, v in audit.items():
    print(f'  {k}: {v}')
if audit['accessed']:
    raise RuntimeError('PM HOLDOUT WAS ACCESSED — contaminated.')
print('\nPM Holdout was not accessed.')